# PyTorch Basics -- Tensors, Autograd, and Neural Networks

Welcome to the PyTorch fundamentals notebook. PyTorch is the dominant framework for
deep learning research and increasingly for production ML. This notebook covers the
three pillars you need before building real models:

1. **Tensors** -- The fundamental data structure (like NumPy arrays, but with GPU support)
2. **Autograd** -- Automatic differentiation that computes gradients for you
3. **nn.Module** -- The building block for neural networks

These concepts underpin everything in the `agentexplorr.classical_ml.deep_learning`
package, from the `SimpleCNN` in `cnn.py` to the `MiniTransformer` in `transformer.py`
to the reusable `Trainer` in `training_loop.py`.

> **Prerequisites**: Basic Python and NumPy. No prior PyTorch experience required.

In [ ]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version:   {np.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

# Determine the best available device
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device:    {device}")

## 1. Tensors -- NumPy Arrays on Steroids

A **tensor** is a multi-dimensional array, just like a NumPy `ndarray`. The two
critical differences are:

| Feature | NumPy | PyTorch Tensor |
|---|---|---|
| **GPU acceleration** | CPU only | CPU *and* GPU (`.to("cuda")`) |
| **Automatic differentiation** | No | Yes (`requires_grad=True`) |

Because tensors track the operations performed on them, PyTorch can automatically
compute gradients -- this is what makes training neural networks possible.

The API deliberately mirrors NumPy, so most operations you know translate directly:
`np.zeros` becomes `torch.zeros`, `np.reshape` becomes `tensor.view`, and so on.

In [ ]:
# --- Creating tensors ---
# From a Python list
t1 = torch.tensor([1.0, 2.0, 3.0])
print(f"From list:    {t1}  (shape={t1.shape}, dtype={t1.dtype})")

# Common constructors (mirror NumPy)
zeros = torch.zeros(2, 3)
ones  = torch.ones(2, 3)
rand  = torch.randn(2, 3)   # standard normal distribution
print(f"\nzeros:\n{zeros}")
print(f"\nrandn:\n{rand}")

# --- NumPy <-> Tensor conversion (zero-copy when possible) ---
np_array = np.array([[1, 2], [3, 4]], dtype=np.float32)
from_np  = torch.from_numpy(np_array)    # shares memory with the numpy array
back_to_np = from_np.numpy()             # shares memory with the tensor
print(f"\nNumPy -> Tensor: {from_np}")
print(f"Tensor -> NumPy: {back_to_np}")

# --- Arithmetic (element-wise, just like NumPy) ---
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print(f"\na + b = {a + b}")
print(f"a * b = {a * b}")           # element-wise multiplication
print(f"a @ b = {a @ b}")           # dot product

# --- Reshaping ---
x = torch.arange(12)
print(f"\nOriginal:  {x}  shape={x.shape}")
print(f"Reshaped:  {x.view(3, 4)}")   # .view() is PyTorch's reshape
print(f"Reshaped:  {x.view(2, 2, 3)}")

# --- Device placement ---
cpu_tensor = torch.randn(3)
print(f"\nDevice: {cpu_tensor.device}")
# If a GPU is available, move the tensor there:
if torch.cuda.is_available():
    gpu_tensor = cpu_tensor.to("cuda")
    print(f"GPU device: {gpu_tensor.device}")

## 2. Autograd -- Automatic Differentiation

Training a neural network boils down to one thing: **computing gradients** so we know
how to adjust each weight to reduce the loss.

PyTorch's **autograd** engine does this automatically. When you set `requires_grad=True`
on a tensor, PyTorch records every operation applied to it in a **computation graph**.
When you call `.backward()` on a scalar result (like a loss), autograd walks this graph
in reverse and fills in `.grad` for every leaf tensor.

This is exactly what happens inside the `Trainer.train_epoch()` method in
`training_loop.py`:

```
loss = loss_fn(model(inputs), targets)   # forward pass -- builds the graph
loss.backward()                          # backward pass -- computes all gradients
optimizer.step()                         # update weights using those gradients
```

In [ ]:
# --- Autograd in action ---
# Let's compute the gradient of a simple function:  y = x^2 + 3x + 1

x = torch.tensor(2.0, requires_grad=True)   # "track this variable"
y = x**2 + 3 * x + 1                        # forward pass -- builds computation graph

print(f"x = {x.item()}")
print(f"y = x^2 + 3x + 1 = {y.item()}")

# Backward pass -- compute dy/dx
y.backward()

# The analytical derivative: dy/dx = 2x + 3.  At x=2: dy/dx = 7
print(f"\ndy/dx (autograd):   {x.grad.item()}")
print(f"dy/dx (analytical): {2 * x.item() + 3}")

# --- Multi-variable example ---
# Simulate a tiny forward pass: loss = sum((w * x_data - y_data)^2)
w = torch.tensor(1.5, requires_grad=True)    # a learnable weight
b = torch.tensor(0.0, requires_grad=True)    # a learnable bias

x_data = torch.tensor([1.0, 2.0, 3.0])
y_data = torch.tensor([2.0, 4.0, 6.0])       # true relationship: y = 2x

# Forward pass
predictions = w * x_data + b
loss = ((predictions - y_data) ** 2).mean()
print(f"\n--- Mini gradient descent step ---")
print(f"Weight={w.item():.2f}, Bias={b.item():.2f}, Loss={loss.item():.4f}")

# Backward pass -- fills in w.grad and b.grad
loss.backward()
print(f"dL/dw = {w.grad.item():.4f}")
print(f"dL/db = {b.grad.item():.4f}")

# A single gradient descent update (learning rate = 0.1)
with torch.no_grad():                         # don't track this update
    w -= 0.1 * w.grad
    b -= 0.1 * b.grad
print(f"After update: Weight={w.item():.4f}, Bias={b.item():.4f}  (closer to w=2, b=0)")

## 3. Building a Neural Network with `nn.Module`

Every PyTorch model inherits from `torch.nn.Module`. The contract is simple:

1. Define your layers in `__init__` (so PyTorch can discover their parameters).
2. Implement `forward()` to describe how data flows through the layers.

That is exactly the pattern used by `SimpleCNN` in `cnn.py` and `MiniTransformer` in
`transformer.py`. Below we build the simplest possible version -- a 2-layer
fully-connected network -- so you can see the pattern clearly before tackling
convolutions or attention heads.

**Key idea**: You never call `backward()` on the model itself. You call it on the
*loss*, and autograd propagates gradients through every `nn.Module` parameter
automatically.

In [ ]:
import torch.nn as nn

# --- Define a simple 2-layer neural network ---
class TwoLayerNet(nn.Module):
    """Input(20) -> Linear(64) -> ReLU -> Linear(3) -> output logits."""

    def __init__(self, input_dim: int = 20, hidden_dim: int = 64, output_dim: int = 3):
        super().__init__()
        # nn.Sequential chains layers in order -- data flows top to bottom
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # weights: (20, 64) + bias: (64,)
            nn.ReLU(),                          # non-linearity: max(0, x)
            nn.Linear(hidden_dim, output_dim),  # weights: (64, 3) + bias: (3,)
        )

    def forward(self, x):
        return self.net(x)


model = TwoLayerNet()
print("Model architecture:")
print(model)

# Count parameters (same pattern used by SimpleCNN.get_num_parameters)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal trainable parameters: {total_params:,}")

# --- Forward pass on random data ---
batch = torch.randn(8, 20)          # 8 samples, 20 features each
logits = model(batch)                # calls model.forward(batch) internally
print(f"\nInput shape:  {batch.shape}")
print(f"Output shape: {logits.shape}")    # (8, 3) -- one score per class

# Convert logits to probabilities
probs = torch.softmax(logits, dim=1)
predicted_classes = probs.argmax(dim=1)
print(f"Probabilities (first sample): {probs[0].detach().numpy().round(3)}")
print(f"Predicted classes: {predicted_classes.tolist()}")

# --- One training step (bringing it all together) ---
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn   = nn.CrossEntropyLoss()
targets   = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1])  # fake labels

optimizer.zero_grad()              # 1. clear old gradients
output  = model(batch)             # 2. forward pass
loss    = loss_fn(output, targets) # 3. compute loss
loss.backward()                    # 4. backward pass (autograd computes gradients)
optimizer.step()                   # 5. update weights

print(f"\nTraining step loss: {loss.item():.4f}")
print("Gradient of first layer weight (shape):", model.net[0].weight.grad.shape)

## Key Takeaways

1. **Tensors** are GPU-capable, gradient-aware arrays. The API mirrors NumPy, so the
   learning curve is gentle.
2. **Autograd** builds a computation graph on the fly and computes all gradients with
   a single `.backward()` call -- no manual calculus required.
3. **`nn.Module`** is the base class for every model. Define layers in `__init__`,
   wire them in `forward()`, and PyTorch handles parameter discovery, device
   placement, and gradient tracking.
4. The **training loop** (`zero_grad -> forward -> loss -> backward -> step`) is the
   universal pattern. The `Trainer` class in `training_loop.py` wraps this with
   validation, early stopping, checkpointing, and learning-rate scheduling.

## Next Steps

- **`cnn.py`** -- See how `SimpleCNN` uses `nn.Conv2d`, `nn.BatchNorm2d`, and
  `nn.MaxPool2d` to classify CIFAR-10 images.
- **`transformer.py`** -- See how `MiniTransformer` builds multi-head self-attention,
  positional encoding, and feed-forward layers from scratch (GPT-style).
- **`training_loop.py`** -- Use the `Trainer` to train any `nn.Module` with early
  stopping, gradient clipping, and checkpoint saving.